# R0 F1 Model Prediction and Forecast

In [12]:
%run ../packages.py
import glob
import fastf1

In [13]:
RAW_PATH = '/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw'
PROCESSED_PATH = "/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/processed"
# PYTENSOR_FLAGS = optimizer = fast_compile

## Create Feature sets

1. rolling feature set of points finishes for each driver / car combination
2. rolling feature set of points finishes for each driver
3. rolling feature set of points finishes for each constructur
4. rolling historical performance for each constructor by race
5. rolling historical performance for each driver  by race

### Rolling feature set of points  finishes for each driver combination

In [14]:
import fastf1
from fastf1 import plotting
import pandas as pd


class F1DataFetcher:
    def __init__(self, cache_dir="cache"):
        # Enable caching to avoid re-downloading data each time
        fastf1.Cache.enable_cache(cache_dir)
        self.cache_dir = cache_dir

    def get_race_results(self, year: int, gp_name: str):
        """Fetch driver finishing positions and points for a given race."""
        session = fastf1.get_session(year, gp_name, 'R')
        session.load()
        results = session.results.copy()
        results['Year'] = year
        results['Race'] = gp_name
        results['RaceDate'] = session.date
        return results.reset_index(drop=True)

    def get_all_races(self, year: int):
        """Fetch all race results for a given season."""
        schedule = fastf1.get_event_schedule(year)
        races = schedule.loc[schedule['EventFormat']
                             == 'conventional']  # skip sprints for now
        all_results = []

        for _, race in races.iterrows():
            try:
                result = self.get_race_results(year, race['EventName'])
                all_results.append(result)
            except Exception as e:
                print(f"⚠️ Skipping {race['EventName']} due to error: {e}")

        return pd.concat(all_results, ignore_index=True)

    def get_driver_points_over_time(self, year: int, driver_abbr: str):
        """Get total points per race for a driver throughout the season."""
        all_results = self.get_all_races(year)
        df = all_results[all_results['Abbreviation'] == driver_abbr]
        df['CumulativePoints'] = df['Points'].cumsum()
        return df[['Year', 'Race', 'Points', 'CumulativePoints']]

# Example usage


f1 = F1DataFetcher(cache_dir=RAW_PATH)
results = f1.get_race_results(2024, "Monaco Grand Prix")
print(results.head())

season_results = f1.get_all_races(2024)
print(season_results.groupby('Abbreviation')['Points'].sum())

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']
core           INFO 	Loading data for Bahrain Grand Prix - Race

  DriverNumber BroadcastName Abbreviation DriverId  TeamName TeamColor  \
0           16     C LECLERC          LEC  leclerc   Ferrari    E80020   
1           81     O PIASTRI          PIA  piastri   McLaren    FF8000   
2           55       C SAINZ          SAI    sainz   Ferrari    E80020   
3            4      L NORRIS          NOR   norris   McLaren    FF8000   
4           63     G RUSSELL          RUS  russell  Mercedes    27F4D2   

     TeamId FirstName LastName         FullName  ...  Q1  Q2  Q3  \
0   ferrari   Charles  Leclerc  Charles Leclerc  ... NaT NaT NaT   
1   mclaren     Oscar  Piastri    Oscar Piastri  ... NaT NaT NaT   
2   ferrari    Carlos    Sainz     Carlos Sainz  ... NaT NaT NaT   
3   mclaren     Lando   Norris     Lando Norris  ... NaT NaT NaT   
4  mercedes    George  Russell   George Russell  ... NaT NaT NaT   

                    Time    Status Points  Laps  Year               Race  \
0 0 days 02:23:15.554000  Finished   25.0  78.0  2024  Monaco Grand Pr

req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req 

Abbreviation
ALB     12.0
ALO     55.0
BEA      7.0
BOT      0.0
COL      4.0
DOO      0.0
GAS     14.0
HAM    184.0
HUL     23.0
LAW      0.0
LEC    247.0
MAG      8.0
NOR    279.0
OCO      4.0
PER     99.0
PIA    214.0
RIC      5.0
RUS    157.0
SAI    201.0
SAR      0.0
STR     24.0
TSU     17.0
VER    280.0
ZHO      0.0
Name: Points, dtype: float64


In [7]:
# fastf1.get_session(2025, 'Monaco Grand Prix', 'R').date
# import theano as tt

In [15]:
from collections import deque
import pandas as pd

# Ensure your DataFrame is sorted chronologically within each group
# season_results = season_results.sort_values(['Abbreviation', 'TeamName', 'RaceDate'])


def rolling_list(series, window=5, include_current=True):
    """
    Returns a list of the last `window` values for each element in `series`.
    If include_current=False, excludes the current row's value.
    """
    d = deque(maxlen=window)
    out = []
    for v in series:
        # append a copy of the current rolling window
        if include_current:
            d.append(v)
            out.append(list(d))
        else:
            out.append(list(d))
            d.append(v)
    return pd.Series(out, index=series.index)


# Apply per driver-constructor
season_results['Points_last5'] = (
    season_results
    .groupby(['Abbreviation', 'TeamName'])['Points']
    .apply(rolling_list, window=5, include_current=False)
    .reset_index(level=[0, 1], drop=True)
)

In [16]:
# # f1_bayes_poisson.py
# # import pymc as pm
# # import pytensor.tensor as pt
# # import arviz as az


# class BayesianF1PointsModelPyMC3:
#     """
#     Hierarchical Poisson (optionally Zero-Inflated) for F1 points at the driver–constructor pair level.
#     Fit once on history, then reuse posterior for next-race and season simulations.
#     """

#     def __init__(self,
#                  num_features=("form5", "qualy_pos",
#                                "track_speed_idx", "weather_rain_prob"),
#                  cat_ids=("pair_id", "constructor_id"),
#                  zero_inflated=False,
#                  draws=2000,
#                  tune=2000,
#                  chains=4,
#                  target_accept=0.9,
#                  random_seed=42):
#         self.num_features = list(num_features)
#         self.cat_ids = list(cat_ids)  # expected: ["pair_id", "constructor_id"]
#         self.zero_inflated = zero_inflated
#         self.draws = draws
#         self.tune = tune
#         self.chains = chains
#         self.target_accept = target_accept
#         self.random_seed = random_seed

#         # learned during fit
#         self.model_ = None
#         self.trace_ = None
#         self.train_means_ = {}
#         self.train_stds_ = {}
#         self.n_pairs_ = None
#         self.n_cons_ = None

#         # PM Data containers
#         self._X_data = None
#         self._y_obs = None
#         self._pair_idx = None
#         self._cons_idx = None

#     # ---------- utilities ----------
#     @staticmethod
#     def _as_int_index(series):
#         # expects already integer ids; factorize strings if needed before calling fit
#         return series.astype("int64").values

#     def _fit_standardizer(self, df):
#         for c in self.num_features:
#             x = df[c].astype(float).values
#             m = x.mean()
#             s = x.std() + 1e-8
#             self.train_means_[c] = m
#             self.train_stds_[c] = s

#     def _transform(self, df):
#         # returns X (N,P) standardized, in the order of self.num_features
#         cols = []
#         for c in self.num_features:
#             x = df[c].astype(float).values
#             m = self.train_means_[c]
#             s = self.train_stds_[c]
#             cols.append((x - m) / s)
#         return np.vstack(cols).T.astype(float)

#     # ---------- public API ----------
#     def fit(self, hist_df):
#         """
#         hist_df columns:
#           target: 'points' (int)
#           ids: 'pair_id', 'constructor_id' (int-coded)
#           numeric features: any in self.num_features (float)
#         """
#         required = {"points", *self.cat_ids, *self.num_features}
#         missing = required - set(hist_df.columns)
#         if missing:
#             raise ValueError(f"Missing required columns: {missing}")

#         # id extents
#         self.n_pairs_ = int(hist_df["pair_id"].max()) + 1
#         self.n_cons_ = int(hist_df["constructor_id"].max()) + 1

#         # standardization fit
#         self._fit_standardizer(hist_df)
#         X = self._transform(hist_df)
#         y = hist_df["points"].astype("int64").values
#         pair_idx = self._as_int_index(hist_df["pair_id"])
#         cons_idx = self._as_int_index(hist_df["constructor_id"])

#         with pm.Model() as model:
#             # data holders for future set_data swaps
#             X_data = pm.Data("X_data", X)
#             y_obs = pm.Data("y_obs", y)
#             pair_data = pm.Data("pair_idx", pair_idx)
#             cons_data = pm.Data("cons_idx", cons_idx)

#             # priors
#             alpha = pm.Normal("alpha", 0.0, 1.5)
#             sigma_pair = pm.HalfNormal("sigma_pair", 1.0)
#             sigma_cons = pm.HalfNormal("sigma_cons", 1.0)
#             pair_re = pm.Normal("pair_re", 0.0, sigma_pair,
#                                 shape=self.n_pairs_)
#             cons_re = pm.Normal("cons_re", 0.0, sigma_cons, shape=self.n_cons_)
#             beta = pm.Normal("beta", 0.0, 1.0, shape=X.shape[1])

#             eta = (alpha + pair_re[pair_data] +
#                    cons_re[cons_data] + pt.dot(X_data, beta))
#             lam = pm.Deterministic("lambda", pt.exp(eta))

#             if not self.zero_inflated:
#                 y_like = pm.Poisson("y_like", mu=lam, observed=y_obs)
#             else:
#                 # simple zero-inflation with intercept-only (extend with features if desired)
#                 z_beta0 = pm.Normal("z_beta0", 0.0, 1.0)
#                 psi = pm.Deterministic("psi", pm.math.sigmoid(z_beta0))
#                 y_like = pm.ZeroInflatedPoisson(
#                     "y_like", psi=psi, theta=lam, observed=y_obs)

#             trace = pm.sample(draws=self.draws, tune=self.tune, chains=self.chains,
#                               target_accept=self.target_accept, random_seed=self.random_seed,
#                               return_inferencedata=True)

#         # cache
#         self.model_ = model
#         self.trace_ = trace
#         self._X_data, self._y_obs = X_data, y_obs
#         self._pair_idx, self._cons_idx = pair_data, cons_data
#         return self

#     def predict_next_race(self, race_df, return_draws=True):
#         """
#         race_df: rows = all driver–constructor pairs in the upcoming race.
#         Must include self.cat_ids + self.num_features (numeric features already engineered).
#         Returns (mu_mean, pred_draws) where:
#           - mu_mean: expected points per row (posterior predictive mean of Poisson mu)
#           - pred_draws: posterior predictive integer draws (n_draws, M), if return_draws=True
#         """
#         if self.model_ is None:
#             raise RuntimeError("Call fit() first.")

#         # transform features with training stats
#         Xr = self._transform(race_df)
#         pair_r = self._as_int_index(race_df["pair_id"])
#         cons_r = self._as_int_index(race_df["constructor_id"])

#         with self.model_:
#             pm.set_data({
#                 "X_data": Xr,
#                 "pair_idx": pair_r,
#                 "cons_idx": cons_r,
#                 "y_obs": np.zeros(len(race_df), dtype="int64")  # dummy
#             })
#             ppc = pm.sample_posterior_predictive(
#                 self.trace_, var_names=["y_like", "lambda"])
#         # shapes:
#         #   ppc["lambda"]: (draws, M)
#         mu = ppc["lambda"].mean(axis=0)
#         if return_draws:
#             # y_like is integer draws (draws, M)
#             return mu, ppc["y_like"]
#         return mu, None

#     def simulate_season(self, future_sched,
#                         # DataFrame with ['pair_id','race_order','points'] if dynamic form
#                         hist_points=None,
#                         k_form=5,
#                         dynamic_form=True,
#                         n_mc=5000,
#                         random_state=0):
#         """
#         future_sched: rows for remaining races with covariates for self.num_features except form5
#                       (we'll set 'form5' each step if dynamic_form=True).
#                       Must include: 'race_id', 'race_order', 'pair_id', 'constructor_id', plus the other numeric regressors.
#         hist_points: if dynamic_form=True, seed last-k windows with actual historical points.
#         Returns:
#           season_summary (DataFrame per pair),
#           season_totals (array [n_pairs, total_draws]) aligned to increasing pair_id.
#         """
#         if self.model_ is None:
#             raise RuntimeError("Call fit() first.")

#         rng = np.random.default_rng(random_state)

#         # Last-k rolling windows for dynamic form
#         last_k = defaultdict(lambda: deque(maxlen=k_form))
#         if dynamic_form and hist_points is not None and not hist_points.empty:
#             for _, row in hist_points.sort_values("race_order").iterrows():
#                 last_k[int(row["pair_id"])].append(float(row["points"]))

#         # collect unique pair ids in schedule to shape outputs
#         pair_ids = np.sort(future_sched["pair_id"].astype(int).unique())
#         idx_map = {p: i for i, p in enumerate(pair_ids)}
#         total_draws = self.trace_.posterior.sizes["draw"] * \
#             self.trace_.posterior.sizes["chain"]
#         # We'll sample posterior predictive per race; to get n_mc draws, we can subset posterior draws.
#         take = min(n_mc, total_draws)
#         season_totals = np.zeros((len(pair_ids), take), dtype=float)

#         # sampling helper to slice posterior draws consistently
#         draw_idx = np.arange(total_draws)
#         rng.shuffle(draw_idx)
#         draw_idx = np.sort(draw_idx[:take])

#         with self.model_:
#             for rid, race in future_sched.sort_values("race_order").groupby("race_id"):
#                 race = race.copy().sort_values("pair_id")
#                 # build/patch form5 dynamically
#                 if dynamic_form and "form5" in self.num_features:
#                     f5 = []
#                     for p in race["pair_id"].astype(int).values:
#                         arr = np.array(last_k[p], dtype=float)
#                         f5.append(arr[-k_form:].mean() if arr.size else 0.0)
#                     race["form5"] = f5

#                 # ensure all numeric features exist
#                 for c in self.num_features:
#                     if c not in race.columns:
#                         # neutral default; better to compute upstream
#                         race[c] = 0.0

#                 Xr = self._transform(race)
#                 pair_r = self._as_int_index(race["pair_id"])
#                 cons_r = self._as_int_index(race["constructor_id"])

#                 pm.set_data({
#                     "X_data": Xr,
#                     "pair_idx": pair_r,
#                     "cons_idx": cons_r,
#                     "y_obs": np.zeros(len(race), dtype="int64")
#                 })
#                 ppc_r = pm.sample_posterior_predictive(
#                     self.trace_, var_names=["y_like"], random_seed=self.random_seed)
#                 # ppc_r["y_like"]: (total_draws, M). Subset to 'take'
#                 yr = ppc_r["y_like"][draw_idx, :]  # (take, M)

#                 # accumulate
#                 for j, p in enumerate(race["pair_id"].astype(int).values):
#                     season_totals[idx_map[p]] += yr[:, j]

#                 # update rolling windows with mean simulated outcome (fast, stable)
#                 if dynamic_form and "form5" in self.num_features:
#                     sim_mean = yr.mean(axis=0)
#                     for j, p in enumerate(race["pair_id"].astype(int).values):
#                         last_k[p].append(sim_mean[j])

#         # summarize
#         rows = []
#         for p in pair_ids:
#             s = season_totals[idx_map[p]]
#             rows.append({
#                 "pair_id": p,
#                 "future_points_mean": float(s.mean()),
#                 "future_points_p5": float(np.percentile(s, 5)),
#                 "future_points_p95": float(np.percentile(s, 95))
#             })
#         summary = pd.DataFrame(rows).sort_values(
#             "future_points_mean", ascending=False)
#         return summary, season_totals, pair_ids

#     # ---------- persistence ----------
#     def save_posterior(self, path_nc: str):
#         if self.trace_ is None:
#             raise RuntimeError("No posterior to save; fit() first.")
#         az.to_netcdf(self.trace_, path_nc)

#     def load_posterior(self, path_nc: str, model_for_set_data_needed=True):
#         # Loading the trace alone is fine for analysis, but for predictions we also need self.model_
#         # (because we use pm.set_data). Usually you'll call fit() once per season and then save.
#         self.trace_ = az.from_netcdf(path_nc)
#         if model_for_set_data_needed and self.model_ is None:
#             raise RuntimeError(
#                 "Trace loaded. To use predict/simulate, reinstantiate and refit (to rebuild the model graph).")

In [34]:
def shrink_mean(lst, mu0, k0):
    # lst is a Python list of points
    s = np.sum(lst) if lst else 0.0
    n = len(lst)
    return (s + k0 * mu0) / (n + k0)


def build_pair_dataset(df, mu0=None, k0=2.0):
    """
    df columns (per your example):
      DriverNumber, Abbreviation, TeamName, Position, Points, Year, Race, Points_last5(list)
    Returns a frame with:
      pair_id, constructor_id, form5, points, race_order (within season), plus optional lags
    """
    df = df.copy()

    # factorize constructor + pair ids (works across full grid, not just one driver)
    df["constructor_id"] = df["TeamName"].astype(
        "category").cat.codes.astype(int)
    df["pair_key"] = df["DriverNumber"].astype(str) + "_" + df["TeamName"]
    df["pair_id"] = df["pair_key"].astype("category").cat.codes.astype(int)

    # season order
    # df["race_order"] = (df["Year"].astype(int).astype(str) + " • " + df["Race"]).astype("category").cat.codes
    df.sort_values("RaceDate", inplace=True)
    # global prior μ0: overall mean points per pair-race if not provided
    if mu0 is None:
        mu0 = float(np.nanmean(df["Points"]))

    # form5 with shrinkage toward μ0 (handles empty lists at season start)
    df["form5"] = df["Points_last5"].apply(lambda L: shrink_mean(L, mu0, k0))

    # optional: create safe lags (no leakage)
    df = df.sort_values(["pair_id", "RaceDate"]).reset_index(drop=True)
    df["points_lag1"] = df.groupby("pair_id")["Points"].shift(1).fillna(mu0)
    df["pos_lag1"] = df.groupby("pair_id")["Position"].shift(
        1).fillna(df["Position"].median())

    # final columns for the Bayesian class
    out = df.rename(columns={"Points": "points"})
    return out

In [35]:
all_drivers_df = build_pair_dataset(season_results)

# season_results.sort_values("RaceDate")

In [36]:
season_results.head().T

,0,1,2,3,4
DriverNumber,1,11,55,16,63
BroadcastName,M VERSTAPPEN,S PEREZ,C SAINZ,C LECLERC,G RUSSELL
Abbreviation,VER,PER,SAI,LEC,RUS
DriverId,max_verstappen,perez,sainz,leclerc,russell
TeamName,Red Bull Racing,Red Bull Racing,Ferrari,Ferrari,Mercedes
TeamColor,3671c6,3671c6,e8002d,e8002d,27f4d2
TeamId,red_bull,red_bull,ferrari,ferrari,mercedes
FirstName,Max,Sergio,Carlos,Charles,George
LastName,Verstappen,Perez,Sainz,Leclerc,Russell
FullName,Max Verstappen,Sergio Perez,Carlos Sainz,Charles Leclerc,George Russell


In [21]:
all_drivers_df.head().T

,0,1,2,3,4
DriverNumber,10,10,10,10,10
BroadcastName,P GASLY,P GASLY,P GASLY,P GASLY,P GASLY
Abbreviation,GAS,GAS,GAS,GAS,GAS
DriverId,gasly,gasly,gasly,gasly,gasly
TeamName,Alpine,Alpine,Alpine,Alpine,Alpine
TeamColor,ff87bc,ff87bc,0093cc,0093cc,0093cc
TeamId,alpine,alpine,alpine,alpine,alpine
FirstName,Pierre,Pierre,Pierre,Pierre,Pierre
LastName,Gasly,Gasly,Gasly,Gasly,Gasly
FullName,Pierre Gasly,Pierre Gasly,Pierre Gasly,Pierre Gasly,Pierre Gasly


In [28]:
# f1_poisson_sklearn.py
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from dataclasses import dataclass, field

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor, LogisticRegression
from sklearn.utils import resample
from joblib import dump, load

# ---------- utilities ----------


def shrink_mean(lst, mu0, k0=2.0):
    """Shrink list-mean toward mu0 with strength k0. Handles empty lists."""
    if isinstance(lst, (list, tuple, np.ndarray)) and len(lst) > 0:
        s, n = float(np.sum(lst)), len(lst)
        return (s + k0 * mu0) / (n + k0)
    return float(mu0)


def prepare_features(df: pd.DataFrame,
                     use_pos_lag=True,
                     k0=2.0,
                     prior_mu=None) -> pd.DataFrame:
    """
    Turns your raw table into model-ready rows:
      - int-coded ids
      - race_order
      - form5 from Points_last5 with shrinkage
      - safe lags (no leakage)
      - target 'points'
    """
    d = df.copy()

    # ids
    d["constructor_id"] = d["TeamName"].astype(
        "category").cat.codes.astype(int)
    d["pair_key"] = d["DriverNumber"].astype(str) + "_" + d["TeamName"]
    d["pair_id"] = d["pair_key"].astype("category").cat.codes.astype(int)

    # race order (replace with your true chronological order if you have a calendar table)
    d["race_order"] = (d["Year"].astype(str) + " • " +
                       d["Race"]).astype("category").cat.codes

    # prior mean for early season
    if prior_mu is None:
        prior_mu = float(pd.to_numeric(d["Points"], errors="coerce").mean())

    # form5
    if "Points_last5" in d.columns:
        d["form5"] = d["Points_last5"].apply(
            lambda L: shrink_mean(L, prior_mu, k0))
    elif "form5" in d.columns:
        d["form5"] = pd.to_numeric(
            d["form5"], errors="coerce").fillna(prior_mu)
    else:
        raise ValueError("Provide either Points_last5 or form5")

    # lags (per pair), no leakage
    d = d.sort_values(["pair_id", "race_order"]).reset_index(drop=True)
    d["points_lag1"] = d.groupby("pair_id")["Points"].shift(1)
    d["points_lag1"] = pd.to_numeric(
        d["points_lag1"], errors="coerce").fillna(prior_mu)

    if use_pos_lag and "Position" in d.columns:
        pos_med = pd.to_numeric(d["Position"], errors="coerce").median()
        d["pos_lag1"] = d.groupby("pair_id")["Position"].shift(1)
        d["pos_lag1"] = pd.to_numeric(
            d["pos_lag1"], errors="coerce").fillna(pos_med)

    d = d.rename(columns={"Points": "points"})
    return d

# ---------- core class ----------


@dataclass
class F1PoissonForecaster:
    num_features: list = field(default_factory=lambda: [
                               "form5", "points_lag1"])
    cat_features: list = field(default_factory=lambda: [
                               "pair_id", "constructor_id"])
    alpha: float = 1e-4                    # L2 strength for PoissonRegressor
    # if True: two-stage (logit -> Poisson)
    hurdle: bool = False
    clip_to_max_points: bool = True        # optional cap to 26 per race
    # >0 enables parameter uncertainty via bootstrapping
    bootstrap_models: int = 0
    max_iter: int = 2000
    random_state: int = 42

    def __post_init__(self):
        self._build_base_pipes()

    def _build_base_pipes(self):
        self.preproc = ColumnTransformer([
            ("num", StandardScaler(), self.num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore",
             sparse=False), self.cat_features)
        ])

        self.poisson = PoissonRegressor(
            alpha=self.alpha, max_iter=self.max_iter)
        self.pipe = Pipeline(
            [("pre", self.preproc), ("poisson", self.poisson)])

        if self.hurdle:
            # score vs no-score head (Bernoulli)
            self.logit = LogisticRegression(
                penalty="l2", C=1.0 / self.alpha if self.alpha > 0 else 1e6,
                max_iter=self.max_iter, solver="lbfgs"
            )
            self.pipe_zero = Pipeline(
                [("pre", self.preproc), ("logit", self.logit)])

        self.ensemble_ = []  # bootstrap models

    # -------------- fitting --------------
    def fit(self, hist: pd.DataFrame):
        X = hist[self.num_features + self.cat_features]
        y = hist["points"].astype(float).values

        # base fit
        if self.hurdle:
            y_bin = (y > 0).astype(int)
            self.pipe_zero.fit(X, y_bin)

            # fit Poisson on positive-only rows
            mask_pos = y > 0
            if mask_pos.sum() == 0:
                raise ValueError(
                    "No positive points in training for hurdle head.")
            self.pipe.fit(X.loc[mask_pos], y[mask_pos])
        else:
            self.pipe.fit(X, y)

        # optional bootstrap ensemble (parameter uncertainty)
        self.ensemble_ = []
        rng = np.random.default_rng(self.random_state)
        for b in range(self.bootstrap_models):
            idx = rng.integers(0, len(hist), size=len(hist))
            Xb, yb = X.iloc[idx], y[idx]
            if self.hurdle:
                yb_bin = (yb > 0).astype(int)
                pipe_zero_b = Pipeline([("pre", self.preproc), ("logit", LogisticRegression(
                    penalty="l2", C=1.0 / self.alpha if self.alpha > 0 else 1e6,
                    max_iter=self.max_iter, solver="lbfgs"))])
                pipe_zero_b.fit(Xb, yb_bin)

                mask_pos_b = yb > 0
                pipe_b = Pipeline([("pre", self.preproc), ("poisson", PoissonRegressor(
                    alpha=self.alpha, max_iter=self.max_iter))])
                if mask_pos_b.any():
                    pipe_b.fit(Xb.loc[mask_pos_b], yb[mask_pos_b])
                else:
                    pipe_b.fit(Xb, yb)  # fallback
                self.ensemble_.append(("hurdle", pipe_zero_b, pipe_b))
            else:
                pipe_b = Pipeline([("pre", self.preproc), ("poisson", PoissonRegressor(
                    alpha=self.alpha, max_iter=self.max_iter))])
                pipe_b.fit(Xb, yb)
                self.ensemble_.append(("plain", pipe_b))

        return self

    # -------------- helpers --------------
    def _predict_mu(self, df: pd.DataFrame) -> np.ndarray:
        X = df[self.num_features + self.cat_features]
        if self.hurdle:
            p_score = self.pipe_zero.predict_proba(X)[:, 1]
            # mean points given scoring
            mu_cond = np.clip(self.pipe.predict(X), 1e-9, None)
            mu = p_score * mu_cond
        else:
            mu = np.clip(self.pipe.predict(X), 1e-9, None)
        if self.clip_to_max_points:
            mu = np.minimum(mu, 26.0)
        return mu

    def _predict_mu_ensemble(self, df: pd.DataFrame, n_members=None) -> np.ndarray:
        """Average μ across bootstrap members + base model."""
        mus = [self._predict_mu(df)]
        if self.ensemble_:
            use = self.ensemble_ if n_members is None else self.ensemble_[
                :n_members]
            X = df[self.num_features + self.cat_features]
            for typ, *pipes in use:
                if typ == "plain":
                    mu = np.clip(pipes[0].predict(X), 1e-9, None)
                else:
                    pipe_zero_b, pipe_b = pipes
                    p_score = pipe_zero_b.predict_proba(X)[:, 1]
                    mu_cond = np.clip(pipe_b.predict(X), 1e-9, None)
                    mu = p_score * mu_cond
                mus.append(np.minimum(mu, 26.0)
                           if self.clip_to_max_points else mu)
        return np.mean(np.vstack(mus), axis=0)

    # -------------- next-race posterior via MC --------------
    def predict_next_race(self, race_df: pd.DataFrame, n_draws=10000, random_state=0):
        """
        Returns (mu_hat, draws) where:
         - mu_hat: expected points per row
         - draws: integer samples (n_rows, n_draws)
        """
        rng = np.random.default_rng(random_state)
        mu = self._predict_mu_ensemble(race_df)
        # Poisson process noise
        draws = rng.poisson(lam=np.clip(mu, 1e-9, None)
                            [:, None], size=(mu.shape[0], n_draws))
        if self.clip_to_max_points:
            draws = np.minimum(draws, 26)
        return mu, draws

    # -------------- season simulation --------------
    def simulate_season(self,
                        schedule_df: pd.DataFrame,
                        # ['pair_id','race_order','points']
                        hist_points: pd.DataFrame = None,
                        k_form=5,
                        dynamic_form=True,
                        n_draws=5000,
                        random_state=0):
        """
        schedule_df: all future pair–race rows with *known* covariates except form5 (we set it if dynamic_form).
                     Must include: pair_id, constructor_id, race_id, race_order, and any extra num_features.
        """
        rng = np.random.default_rng(random_state)

        # rolling last-k points for dynamic form
        last_k = defaultdict(lambda: deque(maxlen=k_form))
        if dynamic_form and hist_points is not None and not hist_points.empty:
            for _, row in hist_points.sort_values("race_order").iterrows():
                last_k[int(row["pair_id"])].append(float(row["points"]))

        # outputs
        pair_ids = np.sort(schedule_df["pair_id"].astype(int).unique())
        pmap = {p: i for i, p in enumerate(pair_ids)}
        season_totals = np.zeros((len(pair_ids), n_draws), dtype=float)

        # iterate race by race
        for rid, race in schedule_df.sort_values("race_order").groupby("race_id"):
            race = race.copy().sort_values("pair_id")

            # build form5 dynamically when requested
            if dynamic_form and "form5" in self.num_features:
                f5 = []
                for p in race["pair_id"].astype(int).values:
                    arr = np.array(last_k[p], dtype=float)
                    f5.append(arr[-k_form:].mean() if arr.size else 0.0)
                race["form5"] = f5

            # ensure all num_features exist
            for c in self.num_features:
                if c not in race.columns:
                    race[c] = 0.0

            mu, draws = self.predict_next_race(
                race, n_draws=n_draws, random_state=rng.integers(1, 1_000_000))

            # accumulate totals
            for j, p in enumerate(race["pair_id"].astype(int).values):
                season_totals[pmap[p]] += draws[j]

            # update rolling windows with mean simulated outcome (stable + fast)
            if dynamic_form and "form5" in self.num_features:
                sim_mean = draws.mean(axis=1)
                for j, p in enumerate(race["pair_id"].astype(int).values):
                    last_k[p].append(sim_mean[j])

        # summarize
        rows = []
        for p in pair_ids:
            s = season_totals[pmap[p]]
            rows.append({
                "pair_id": p,
                "future_points_mean": float(s.mean()),
                "future_points_p5": float(np.percentile(s, 5)),
                "future_points_p95": float(np.percentile(s, 95)),
            })
        summary = pd.DataFrame(rows).sort_values(
            "future_points_mean", ascending=False)
        return summary, season_totals, pair_ids

    # -------------- persistence --------------
    def save(self, path: str):
        dump({
            "pipe": self.pipe,
            "pipe_zero": getattr(self, "pipe_zero", None),
            "ensemble": self.ensemble_,
            "config": {
                "num_features": self.num_features,
                "cat_features": self.cat_features,
                "alpha": self.alpha,
                "hurdle": self.hurdle,
                "clip_to_max_points": self.clip_to_max_points,
                "bootstrap_models": self.bootstrap_models,
                "max_iter": self.max_iter,
                "random_state": self.random_state
            }
        }, path)

    @staticmethod
    def load(path: str):
        obj = load(path)
        cfg = obj["config"]
        f = F1PoissonForecaster(**cfg)
        f.pipe = obj["pipe"]
        f.pipe_zero = obj["pipe_zero"]
        f.ensemble_ = obj["ensemble"]
        return f

In [29]:
# 1) Prep your historical data (your example structure works)
# converts Points_last5 -> form5, adds lags, ids
hist = prepare_features(all_drivers_df, k0=2.0)

# 2) Fit a plain Poisson or a hurdle model
#    Start plain; if zeros are excessive, set hurdle=True
fore = F1PoissonForecaster(
    # add "pos_lag1" if you’re okay using last race position
    num_features=["form5", "points_lag1"],
    cat_features=["pair_id", "constructor_id"],
    alpha=1e-4,
    hurdle=False,
    # optional: parameter uncertainty via bootstrapping
    bootstrap_models=50,
    random_state=123
).fit(hist)

# 3) Next-race distribution for the full grid (or a single driver)
# next_race_df must have the same feature columns; if dynamic_form, you can leave form5 blank
mu_next, draws_next = fore.predict_next_race(next_race_df, n_draws=10000)

# 4) Season simulation (static or dynamic form)
season_summary, season_totals, pair_ids = fore.simulate_season(
    # rows: one per pair–race with known covariates; include race_id, race_order
    schedule_df=future_sched_df,
    hist_points=hist[["pair_id", "race_order", "points"]],
    k_form=5,
    dynamic_form=True,
    n_draws=5000,
    random_state=7
)

KeyError: 'Points'

NameError: name 'BayesianF1PointsModelPyMC3' is not defined

In [13]:
# season_results[season_results['Abbreviation'] == 'LEC']
import arviz as az

az.summary(m.trace_, var_names=["alpha", "sigma_pair", "sigma_cons", "beta"])
az.plot_posterior(m.trace_, var_names=["beta"])

ValueError: Can only convert xarray dataarray, xarray dataset, dict, pytree (if 'dm-tree' is installed), netcdf filename, numpy array, pystan fit, emcee fit, pyro mcmc fit, numpyro mcmc fit, cmdstan fit csv filename, cmdstanpy fit to InferenceData, not NoneType

In [10]:
results_2024 = pd.read_csv(f"{PROCESSED_PATH}/2024/results.csv")
results_2025 = pd.read_csv(f"{PROCESSED_PATH}/2025/results.csv")
results_2024.head()

,Pos.,Driver,Constructor,Time/Retired,Grid,Laps,Points,race_year,race
0,1,#1Max Verstappen,Red Bull,1:54:23.566,1st,53.0,26.0,2024,japanese
1,2,#11Sergio Pérez,Red Bull,+12.535,2nd,53.0,18.0,2024,japanese
2,3,#55Carlos Sainz Jr.,Ferrari,+20.866,4th,53.0,15.0,2024,japanese
3,4,#16Charles Leclerc,Ferrari,+26.522,8th,53.0,12.0,2024,japanese
4,5,#4Lando Norris,McLaren,+29.700,3rd,53.0,10.0,2024,japanese


In [12]:
results_2024.head(25)

,Pos.,Driver,Constructor,Time/Retired,Grid,Laps,Points,race_year,race
0,1,#1Max Verstappen,Red Bull,1:54:23.566,1st,53.0,26.0,2024,japanese
1,2,#11Sergio Pérez,Red Bull,+12.535,2nd,53.0,18.0,2024,japanese
2,3,#55Carlos Sainz Jr.,Ferrari,+20.866,4th,53.0,15.0,2024,japanese
3,4,#16Charles Leclerc,Ferrari,+26.522,8th,53.0,12.0,2024,japanese
4,5,#4Lando Norris,McLaren,+29.700,3rd,53.0,10.0,2024,japanese
5,6,#14Fernando Alonso,Aston Martin,+44.272,5th,53.0,8.0,2024,japanese
6,7,#63George Russell,Mercedes,+45.951,9th,53.0,6.0,2024,japanese
7,8,#81Oscar Piastri,McLaren,+47.525,6th,53.0,4.0,2024,japanese
8,9,#44Lewis Hamilton,Mercedes,+48.626,7th,53.0,2.0,2024,japanese
9,10,#22Yuki Tsunoda,RB,+1 lap,10th,52.0,1.0,2024,japanese


In [ ]:
def rolling_points():
    return


## Feature Iteration Infrastructure

Tools for rapidly adding/removing features and measuring their impact via walk-forward evaluation.

**Workflow:**
1. Run **Multi-Year Data Loader** to build `multi_df` from the fastf1 cache
2. Toggle features in **Feature Registry** (`active: True/False`)
3. Run **Walk-Forward Eval** to see Spearman / MAE / top-3 accuracy across 2022–2024
4. Run **Feature Ablation** to see which features are load-bearing
5. Run **Experiment Log** to persist results between sessions


In [ ]:

# ── Multi-Year Data Loader ────────────────────────────────────────────────────
# Uses the fastf1 cache (ff1pkl files) — fast if already cached.
# Adds grid_position, dnf_rate5, track_form5, season_rank alongside
# the existing form5/points_lag1 features.


def load_multiyear_fastf1(fetcher, years=range(2020, 2025)):
    """Load race results for multiple seasons from the fastf1 cache."""
    dfs = []
    for year in years:
        try:
            df = fetcher.get_all_races(year)
            dfs.append(df)
            print(f"  ✓ {year}: {len(df)} driver-race rows")
        except Exception as e:
            print(f"  ⚠ {year}: {e}")
    return pd.concat(dfs, ignore_index=True)


def build_multiyear_features(season_df, k0=2.0):
    """
    Build the full feature set from a fastf1 results DataFrame.
    Expects columns: Abbreviation, TeamName, Points, GridPosition,
                     Status, Year, Race, RaceDate
    """
    df = season_df.copy()
    df["Points"] = pd.to_numeric(df["Points"], errors="coerce").fillna(0)
    df["GridPosition"] = pd.to_numeric(df["GridPosition"], errors="coerce").fillna(10)
    df = df.sort_values(["Abbreviation", "RaceDate"]).reset_index(drop=True)
    mu0 = float(df["Points"].mean())

    # ── rolling form (no leakage) ──
    df["Points_last5"] = (
        df.groupby("Abbreviation")["Points"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=0, drop=True)
    )
    df["form5"] = df["Points_last5"].apply(lambda L: shrink_mean(L, mu0, k0))
    df["points_lag1"] = df.groupby("Abbreviation")["Points"].shift(1).fillna(mu0)

    # ── grid position ──
    df["grid_position"] = df["GridPosition"]

    # ── DNF rate: classified if Finished or lapped (+N Laps) ──
    classified = (
        df["Status"].str.startswith("Finished", na=False) |
        df["Status"].str.startswith("+", na=False)
    )
    df["dnf"] = (~classified).astype(int)
    df["dnf_rate5"] = (
        df.groupby("Abbreviation")["dnf"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=0, drop=True)
        .apply(lambda L: float(np.mean(L)) if len(L) > 0 else 0.0)
    )

    # ── track-specific form ──
    track = df.sort_values(["Abbreviation", "Race", "RaceDate"]).copy()
    track["_track_last5"] = (
        track.groupby(["Abbreviation", "Race"])["Points"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=[0, 1], drop=True)
    )
    track["track_form5"] = track["_track_last5"].apply(
        lambda L: float(np.mean(L)) if isinstance(L, list) and len(L) > 0 else mu0
    )
    df["track_form5"] = track["track_form5"].reindex(df.index)

    # ── season rank entering race ──
    df = df.sort_values(["Year", "RaceDate", "Abbreviation"]).reset_index(drop=True)
    df["_cum"] = df.groupby(["Year", "Abbreviation"])["Points"].cumsum()
    df["_cum_lag"] = df.groupby(["Year", "Abbreviation"])["_cum"].shift(1).fillna(0)
    df["season_rank"] = (
        df.groupby(["Year", "RaceDate"])["_cum_lag"]
        .rank(ascending=False, method="min")
    )
    df.drop(columns=["_cum", "_cum_lag"], inplace=True)

    # ── categorical IDs ──
    df["constructor_id"] = df["TeamName"].astype("category").cat.codes.astype(int)
    df["pair_id"] = (df["Abbreviation"] + "_" + df["TeamName"]).astype("category").cat.codes.astype(int)

    # lowercase target for F1PoissonForecaster
    df["points"] = df["Points"]
    return df


print("Loading multi-year data from fastf1 cache...")
multi_raw = load_multiyear_fastf1(f1, years=range(2020, 2025))
multi_df = build_multiyear_features(multi_raw)
print(f"\nDataset: {multi_df.shape[0]} driver-race rows across {multi_df['Year'].nunique()} seasons")
multi_df[["Abbreviation", "Race", "Year", "form5", "grid_position", "dnf_rate5", "track_form5", "season_rank"]].head(8)


In [ ]:

# ── Feature Registry ──────────────────────────────────────────────────────────
# Toggle features on/off here — no other code needs to change.

FEATURE_CONFIG = {
    # Baseline (already implemented)
    "form5":         {"active": True,  "type": "numeric"},
    "points_lag1":   {"active": True,  "type": "numeric"},
    "pair_id":       {"active": True,  "type": "categorical"},
    "constructor_id":{"active": True,  "type": "categorical"},
    # New candidates — flip active: True to include, then rerun walk_forward_eval
    "grid_position": {"active": False, "type": "numeric"},   # qualifying/grid start
    "dnf_rate5":     {"active": False, "type": "numeric"},   # rolling DNF rate
    "track_form5":   {"active": False, "type": "numeric"},   # avg points at this circuit
    "season_rank":   {"active": False, "type": "numeric"},   # current WDC standing
}

active_numeric = [k for k, v in FEATURE_CONFIG.items() if v["active"] and v["type"] == "numeric"]
active_cat     = [k for k, v in FEATURE_CONFIG.items() if v["active"] and v["type"] == "categorical"]
print("Numeric :", active_numeric)
print("Categorical:", active_cat)


In [ ]:

# ── Walk-Forward Evaluation Harness ──────────────────────────────────────────
# Train on all seasons < test_year, evaluate on test_year.
# Use this to compare any feature set change in seconds.

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error


def top3_accuracy(df: pd.DataFrame, preds: np.ndarray) -> float:
    """Fraction of actual podium (top-3 points scorers) predicted in model's top 3."""
    df = df.copy()
    df["_pred"] = preds
    hits, total = 0, 0
    for _, race in df.groupby(["Year", "Race"]):
        actual_top3 = set(race.nlargest(3, "points").index)
        pred_top3 = set(race.nlargest(3, "_pred").index)
        hits += len(actual_top3 & pred_top3)
        total += 3
    return hits / total if total > 0 else 0.0


def walk_forward_eval(df, num_feats, cat_feats, test_seasons=(2022, 2023, 2024)):
    """Train on years < test_year, score on test_year. Returns per-season metrics."""
    rows = []
    required = num_feats + cat_feats + ["points"]
    for test_year in test_seasons:
        train = df[df["Year"] < test_year].dropna(subset=required)
        test = df[df["Year"] == test_year].dropna(subset=required)
        if train.empty or test.empty:
            continue
        model = F1PoissonForecaster(num_features=num_feats, cat_features=cat_feats)
        model.fit(train)
        preds = model._predict_mu(test)
        rows.append({
            "season": test_year,
            "spearman": round(spearmanr(test["points"], preds).statistic, 4),
            "mae": round(mean_absolute_error(test["points"], preds), 4),
            "top3_acc": round(top3_accuracy(test, preds), 4),
        })
    result = pd.DataFrame(rows)
    display(result)
    print("Mean:", result[["spearman", "mae", "top3_acc"]].mean().round(4).to_dict())
    return result


# Baseline run
baseline = walk_forward_eval(multi_df, active_numeric, active_cat)


In [ ]:

# ── Feature Ablation ──────────────────────────────────────────────────────────
# Remove one feature at a time and measure the Spearman drop.
# Run after setting FEATURE_CONFIG and loading multi_df.

baseline_scores = walk_forward_eval(multi_df, active_numeric, active_cat)
baseline_spearman = baseline_scores["spearman"].mean()
print(f"Baseline mean Spearman: {baseline_spearman:.4f}\n")

ablation = {}
for feat in active_numeric + active_cat:
    n_feats = [f for f in active_numeric if f != feat]
    c_feats = [f for f in active_cat if f != feat]
    res = walk_forward_eval(multi_df, n_feats, c_feats)
    delta = round(res["spearman"].mean() - baseline_spearman, 4)
    ablation[feat] = delta
    print(f"  -{feat}: Δ={delta:+.4f}")

ax = pd.Series(ablation).sort_values().plot(
    kind="barh", figsize=(7, 4),
    title="Spearman drop when feature removed (more negative = more important)",
    xlabel="ΔSpearman vs. baseline"
)
ax.axvline(0, color="black", linewidth=0.8)


In [ ]:

# ── Experiment Log ────────────────────────────────────────────────────────────
# Appends the current feature set's eval scores to a CSV so nothing is lost
# between sessions. Call this after any promising feature change.

import os

LOG_PATH = f"{PROCESSED_PATH}/experiment_log.csv"


def log_experiment(df, num_feats, cat_feats, note=""):
    """Run walk-forward eval and append results to the experiment log CSV."""
    run = walk_forward_eval(df, num_feats, cat_feats)
    run["features"] = str(sorted(num_feats + cat_feats))
    run["note"] = note

    if os.path.exists(LOG_PATH):
        existing = pd.read_csv(LOG_PATH)
        log = pd.concat([existing, run], ignore_index=True)
    else:
        log = run

    log.to_csv(LOG_PATH, index=False)

    summary = (
        log.groupby("features")[["spearman", "mae", "top3_acc"]]
        .mean()
        .round(4)
        .sort_values("spearman", ascending=False)
    )
    display(summary)
    return log


# Example — call with a note describing what you changed:
# log_experiment(multi_df, active_numeric, active_cat, note="baseline: form5 + points_lag1")
